# Phase 2 演習 — テンソル・主値・不変量

対応する本文：[`docs/texts/phase2-tensors-invariants.md`](../docs/texts/phase2-tensors-invariants.md)

このNotebookは提出用です。各 **あなたの回答** を埋め、コードセルを実行してください。数値だけでなく、座標系・符号規約・適用範囲を記述します。

## 提出情報

- 氏名・日付：
- 実行環境（任意）：

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

---

## 演習 1 — 座標を回しても応力状態は同じか

単位を MPa とし、$\boldsymbol{\sigma}=\mathrm{diag}(100,0,0)$ を考えます。$z$ 軸まわりに $45^\circ$ 回転した基底での成分を、本文の $\boldsymbol{\sigma}'=\mathbf{Q}^T\boldsymbol{\sigma}\mathbf{Q}$ で求めます。

1. 下の結果の $\sigma'_{xx},\sigma'_{yy},\sigma'_{xy}$ を書く。
2. せん断成分が現れても、新しい荷重が生じたわけではない理由を3〜5文で説明する。

### あなたの回答（演習 1）

1. $\sigma'_{xx} = 50, \sigma'_{yy} = 50, \sigma'_{xy} = -50$
2. $\boldsymbol{\sigma}'$ は $\boldsymbol{\sigma}$ を異なる座標系で記述しただけであり、同じ応力状態を表している。$\boldsymbol{\sigma}'$ の主応力は 0.0, 0.0, 100.0 と(数値誤差の範囲で)一致している。これは $\boldsymbol{\sigma}$ の非対角成分がゼロ、すなわち主軸での表現であることを踏まえると、両者から導かれる主応力が一致していることを示している。

In [2]:
theta = np.deg2rad(45.0)
Q = np.array([[np.cos(theta), -np.sin(theta), 0.0],
              [np.sin(theta),  np.cos(theta), 0.0],
              [0.0,            0.0,           1.0]])
sigma = np.diag([100.0, 0.0, 0.0])
sigma_rot = Q.T @ sigma @ Q
print(sigma_rot, 'MPa')
assert np.allclose(np.linalg.eigvalsh(sigma_rot), [0.0, 0.0, 100.0])

[[ 50. -50.   0.]
 [-50.  50.   0.]
 [  0.   0.   0.]] MPa


---

## 演習 2 — 主応力と最大せん断応力

次の対称応力テンソル（MPa）を考えます。

$$\boldsymbol{\sigma}=\begin{bmatrix}120&50&0\\50&20&0\\0&0&10\end{bmatrix}$$

1. 主応力を大きい順に書く。
2. 最大せん断応力を求める。
3. 主軸座標でせん断成分がゼロになる理由を、固有値・固有ベクトルという語を使って説明する。

### あなたの回答（演習 2）

1. 141, 10, -0.711 (MPa)
2. 71 MPa
3. 応力テンソルは主応力を固有値としてもつが、応力テンソルが実対称行列として表現できることから、3つの主応力に対応する固有ベクトルは直交し、これを主軸として選択することができる。したがって、主軸座標で表された応力テンソルは対角かされ、非対角成分、すなわちせん断成分はゼロになる。

In [3]:
sigma = np.array([[120.0, 50.0, 0.0],
                  [ 50.0, 20.0, 0.0],
                  [  0.0,  0.0, 10.0]])
principal_asc, Qp = np.linalg.eigh(sigma)
principal = principal_asc[::-1]
tau_max = (principal[0] - principal[-1]) / 2.0
print('principal stresses [MPa] =', principal)
print('tau_max [MPa] =', tau_max)
print('Q^T sigma Q [MPa] =\n', Qp.T @ sigma @ Qp)

principal stresses [MPa] = [140.710678  10.        -0.710678]
tau_max [MPa] = 70.71067811865476
Q^T sigma Q [MPa] =
 [[ -0.710678   0.         0.      ]
 [  0.        10.         0.      ]
 [ -0.         0.       140.710678]]


---

## 演習 3 — 静水圧成分と偏差成分

$\boldsymbol{\sigma}=\mathrm{diag}(160,40,-20)$ MPa とします。引張正の規約を使います。

1. 平均応力 $p$ と偏差応力テンソル $\mathbf{s}$ を求める。
2. $\mathrm{tr}(\mathbf{s})=0$ を確認する。
3. $100\mathbf{I}$ MPa の von Mises 応力がゼロであっても「応力ゼロ」ではない理由を説明する。

### あなたの回答（演習 3）

1. $p = 60$ MPa, $\mathbf{s} = \mathrm{diag}(100, -20, -80)$ MPa
2. $\mathrm{tr}(\mathbf{s}) = 100 - 20 - 80 = 0$
3. 体積変化に寄与する平均応力 $p = 100$ MPa が存在する。

In [4]:
sigma = np.diag([160.0, 40.0, -20.0])
p = np.trace(sigma) / 3.0
s = sigma - p * np.eye(3)
print('p [MPa] =', p)
print('s [MPa] =\n', s)
print('trace(s) [MPa] =', np.trace(s))
assert np.isclose(np.trace(s), 0.0)

p [MPa] = 60.0
s [MPa] =
 [[100.   0.   0.]
 [  0. -20.   0.]
 [  0.   0. -80.]]
trace(s) [MPa] = 0.0


---

## 演習 4 — von Mises 応力を二通りで計算する

演習3の応力について、(a) $\sqrt{3J_2}$、(b) 主応力差の式、の二通りで von Mises 応力を計算します。

1. 二つの結果が一致することを確認する。
2. この相当応力を材料の降伏応力と比較してよい仮定を二つ、比較だけでは不十分な場合を一つ挙げる。

### あなたの回答（演習 4）

1. `np.isclose(vm_from_j2, vm_from_principal) = True` であることから、二つの結果は一致する。
2. 材料モデルがJ2型である場合、延性降伏が発生すると考えられる場合は、相当応力と降伏応力を比較できる。一方で、き裂や層間破壊の発生については、相当応力だけでは議論できず、主応力や応力成分など、応力の方向に関する情報を考慮する必要がある。

In [5]:
J2 = 0.5 * np.sum(s * s)
vm_from_j2 = np.sqrt(3.0 * J2)
sig = np.linalg.eigvalsh(sigma)
vm_from_principal = np.sqrt(((sig[0]-sig[1])**2 + (sig[1]-sig[2])**2 + (sig[2]-sig[0])**2) / 2.0)
print('J2 [MPa^2] =', J2)
print('von Mises from J2 [MPa] =', vm_from_j2)
print('von Mises from principal stresses [MPa] =', vm_from_principal)
assert np.isclose(vm_from_j2, vm_from_principal)

J2 [MPa^2] = 8400.0
von Mises from J2 [MPa] = 158.74507866387543
von Mises from principal stresses [MPa] = 158.74507866387543


---

## 結果レビューと振り返り

手元のCAE結果または想定したブラケット解析について記入してください。

- 比較した相当応力と材料値の単位：MPa
- 出力位置（積分点、要素中心、節点平均など）：節点平均
- J2/von Mises を使う根拠と使えない可能性：材料がアルミ(延性)であるブラケットに曲げ荷重をかけた際の変形を調べるため
- 主応力または応力成分も確認すべき箇所：剛体との接合部
- メッシュ・拘束・接触について追加で確認すること：要素長に対する応力の収束

今回、最も理解が変わった概念：

まだ説明できない／導出したい概念：

教材または演習の改善提案：

この内容は `docs/learning-log.md` の Phase 2 項目へ追記してください。